In [2]:
import pandas as pd 
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib

In [6]:
hate_dataset = load_dataset("cardiffnlp/tweet_eval", "hate", split="train")
print(hate_dataset)

README.md:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

hate/train-00000-of-00001.parquet:   0%|          | 0.00/816k [00:00<?, ?B/s]

hate/test-00000-of-00001.parquet:   0%|          | 0.00/278k [00:00<?, ?B/s]

hate/validation-00000-of-00001.parquet:   0%|          | 0.00/103k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2970 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label'],
    num_rows: 9000
})


In [7]:
df_hate = pd.DataFrame(hate_dataset)
print(df_hate.head())
print("\nLabel distribution:")
print(df_hate['label'].value_counts())

                                                text  label
0  @user nice new signage. Are you not concerned ...      0
1  A woman who you fucked multiple times saying y...      1
2  @user @user real talk do you have eyes or were...      1
3  your girlfriend lookin at me like a groupie in...      1
4                        Hysterical woman like @user      0

Label distribution:
label
0    5217
1    3783
Name: count, dtype: int64


In [8]:
X = df_hate['text']
y = df_hate['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

vectorizer_toxic = TfidfVectorizer(
    stop_words='english',
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_tfidf = vectorizer_toxic.fit_transform(X_train)
X_test_tfidf = vectorizer_toxic.transform(X_test)

model_toxic = LogisticRegression(max_iter=1000)
model_toxic.fit(X_train_tfidf, y_train)

print(classification_report(y_test, model_toxic.predict(X_test_tfidf)))

              precision    recall  f1-score   support

           0       0.78      0.90      0.83      1036
           1       0.83      0.65      0.73       764

    accuracy                           0.79      1800
   macro avg       0.80      0.78      0.78      1800
weighted avg       0.80      0.79      0.79      1800



In [9]:
joblib.dump(model_toxic, 'toxic_model.pkl')
joblib.dump(vectorizer_toxic, 'vectorizer_toxic.pkl')
print("Toxic model saved.")

Toxic model saved.
